# Build the study cohort

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


Provide the study's excluded patient IDs as a JSON list through `SMMNLP_EXCLUSIONS_FILE`. The example configuration contains an empty list.


In [ ]:
import logging
import random

import pandas as pd

from smmnlp.paths import ProjectPaths

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Read in raw texts from the 2nd cohort (N=237)


In [ ]:
readfile = paths.data("Second237/Data/CISNET_SMMNLP_237_120day.json")
cohort2 = pd.read_json(readfile)

In [ ]:
len(cohort2)

In [ ]:
cohort2.head(5)

In [ ]:
exclude_SSN = paths.load_excluded_patient_ids()
cohort2_select = cohort2[~cohort2["PatientSSN"].isin(exclude_SSN)]
len(cohort2_select)

In [ ]:
cohort2_select["index_year"] = cohort2_select["index_date"].str[5:]
cohort2_select["cohort_version"] = "S2"

In [ ]:
cohort2_select.head(5)

In [ ]:
cohort2_select2 = cohort2_select[["PatientSSN", "index_year"]]
uniquessn = cohort2_select2.drop_duplicates()

In [ ]:
print(uniquessn["index_year"].value_counts(dropna=False))

In [ ]:
sizes = {"2022": 19, "2023": 39, "2024": 42}
cohort2_final = uniquessn.groupby("index_year", group_keys=False).apply(
    lambda x: x.sample(n=sizes[x.name], random_state=76)
)

In [ ]:
len(cohort2_final)

In [ ]:
cohort2_100 = cohort2_select[cohort2_select["PatientSSN"].isin(cohort2_final["PatientSSN"])]

In [ ]:
len(cohort2_100)

In [ ]:
print(cohort2_100["index_year"].value_counts(dropna=False))

In [ ]:
savefile_name = paths.data("Second237/Data/CISNET_SMMNLP_237_120day_Select100.json")
cohort2_100.to_json(savefile_name)

In [ ]:
savefile_name = paths.data("Second237/Data/CISNET_SMMNLP_237_120day_Select100.xlsx")
cohort2_100.to_excel(savefile_name, engine="xlsxwriter", sheet_name="Sheet1", index=False)

In [ ]:
readfiletrue = paths.data("Second237/True/SecondCohort237_Complete.xlsx")
true = pd.read_excel(readfiletrue)

In [ ]:
len(true)

In [ ]:
cohort2_100_true = true[true["PatientSSN"].isin(cohort2_final["PatientSSN"])]

In [ ]:
len(cohort2_100_true)

In [ ]:
cohort2_100_true.head()

In [ ]:
savefile_name = paths.data("Second237/True/SecondCohort237_Select100.xlsx")
cohort2_100_true.to_excel(savefile_name, engine="xlsxwriter", sheet_name="Sheet1", index=False)

In [ ]:
unique_ids = cohort2_100["PatientSSN"].unique()
train_size = int(len(unique_ids) * 0.7)
random.seed(66695)
train_ids = random.sample(list(unique_ids), train_size)
test_ids = [PatientSSN for PatientSSN in unique_ids if PatientSSN not in train_ids]
train_df_s2 = cohort2_100[cohort2_100["PatientSSN"].isin(train_ids)]
test_df_s2 = cohort2_100[cohort2_100["PatientSSN"].isin(test_ids)]

In [ ]:
train_df_s2.head(5)

In [ ]:
len(train_df_s2)

In [ ]:
len(test_df_s2)

## Read in raw texts from the 1st cohort (N=500)


In [ ]:
readfile = paths.data("Random500/Update/CISNET_SMMNLP_500_120day.json")
cohort1 = pd.read_json(readfile)
cohort1_select = cohort1[
    ["PatientSSN", "NoteDate", "NoteText", "Pt_index", "Note_index", "index_date"]
]

In [ ]:
len(cohort1_select)

In [ ]:
cohort1_select["index_year"] = cohort1_select["index_date"].str[5:]
cohort1_select["cohort_version"] = "S1"

In [ ]:
cohort1_select.head()

In [ ]:
unique_ids = cohort1_select["PatientSSN"].unique()
train_size = int(len(unique_ids) * 0.7)
random.seed(56)
train_ids = random.sample(list(unique_ids), train_size)
test_ids = [PatientSSN for PatientSSN in unique_ids if PatientSSN not in train_ids]
train_df_s1 = cohort1_select[cohort1_select["PatientSSN"].isin(train_ids)]
test_df_s1 = cohort1_select[cohort1_select["PatientSSN"].isin(test_ids)]

In [ ]:
len(train_df_s1)

In [ ]:
len(test_df_s1)

## Combine S1 and S2 together


In [ ]:
training_df = pd.concat([train_df_s1, train_df_s2], ignore_index=True)

In [ ]:
testing_df = pd.concat([test_df_s1, test_df_s2], ignore_index=True)

In [ ]:
len(training_df)

In [ ]:
len(testing_df)

In [ ]:
savefile_name1 = paths.data("Update/Data/Training/SMMNLP_training_data.json")
training_df.to_json(savefile_name1)
savefile_name2 = paths.data("Update/Data/Testing/SMMNLP_testing_data.json")
testing_df.to_json(savefile_name2)

In [ ]:
savefile_name1 = paths.data("Update/Data/Training/SMMNLP_training_data.xlsx")
training_df.to_excel(savefile_name1, engine="xlsxwriter", sheet_name="Sheet1", index=False)
savefile_name2 = paths.data("Update/Data/Testing/SMMNLP_testing_data.xlsx")
testing_df.to_excel(savefile_name2, engine="xlsxwriter", sheet_name="Sheet1", index=False)

## Read in true labels


In [ ]:
readfiletrue = paths.data("Random500/Update/FirstCohort500_Complete.xlsx")
true_s1 = pd.read_excel(readfiletrue)

In [ ]:
len(true_s1)

In [ ]:
true_s1.head()

In [ ]:
train_df_s1.head()

In [ ]:
train_df_true_s1 = train_df_s1.merge(
    true_s1[["PatientSSN", "Note_index", "SMM Dx (final)"]],
    on=["PatientSSN", "Note_index"],
    how="inner",
)
test_df_true_s1 = test_df_s1.merge(
    true_s1[["PatientSSN", "Note_index", "SMM Dx (final)"]],
    on=["PatientSSN", "Note_index"],
    how="inner",
)

In [ ]:
len(train_df_true_s1)

In [ ]:
len(test_df_true_s1)

In [ ]:
train_df_true_s1.head()

In [ ]:
readfiletrue = paths.data("Second237/True/SecondCohort237_Complete.xlsx")
true_s2 = pd.read_excel(readfiletrue)

In [ ]:
len(true_s2)

In [ ]:
true_s2.head()

In [ ]:
train_df_s2.head()

In [ ]:
train_df_true_s2 = train_df_s2.merge(
    true_s2[["PatientSSN", "Note_index", "SMM Dx (final)"]],
    on=["PatientSSN", "Note_index"],
    how="inner",
)
test_df_true_s2 = test_df_s2.merge(
    true_s2[["PatientSSN", "Note_index", "SMM Dx (final)"]],
    on=["PatientSSN", "Note_index"],
    how="inner",
)

In [ ]:
len(train_df_true_s2)

In [ ]:
len(test_df_true_s2)

## Combine S1 and S2 together


In [ ]:
training_df_true = pd.concat([train_df_true_s1, train_df_true_s2], ignore_index=True)

In [ ]:
testing_df_true = pd.concat([test_df_true_s1, test_df_true_s2], ignore_index=True)

In [ ]:
savefile_name1 = paths.data("Update/True/SMMNLP_training_true_phase1.xlsx")
training_df_true.to_excel(savefile_name1, engine="xlsxwriter", sheet_name="Sheet1", index=False)
savefile_name2 = paths.data("Update/True/SMMNLP_testing_true_phase1.xlsx")
testing_df_true.to_excel(savefile_name2, engine="xlsxwriter", sheet_name="Sheet1", index=False)

In [ ]:
testing_df_true.head()